In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込みエラー: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 28. データ並列

> **学習目標**
> - データ (All-Reduce)
> -
> - DDP (DistributedDataParallel)

## 28.1 GPU

LLM 学習 · :
- 7B モデル FP16 + AdamW = ~70GB
- =
- 速度: 1 GPU

: GPU .

## 28.2 データ

1. モデル $N$ GPU
2. データ $N$ GPU
3. GPU データ 計算
4. → GPU
5. GPU → モデル

:
- : $\mathbf{g}_i = \nabla \mathcal{L}_i(\theta)$
- : $\mathbf{g} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{g}_i$
- : $\theta \leftarrow \theta - \eta \mathbf{g}$

 : $B_{\text{eff}} = N \cdot B_{\text{local}}$


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# データ   ( GPU gradient accumulation )
class DataParallelSimulator:
    """ GPU N-GPU DDP  ."""
    def __init__(self, model, n_gpus=4):
        self.model = model
        self.n_gpus = n_gpus
        # N " GPU" =  Model  (gradient accumulation )
        #   Model gradient n_gpus 
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    def step(self, x_batch, y_batch):
        """x_batch: (N*B, ...), N GPU  B Sample."""
        # 1. Batch N
        chunks = torch.chunk(x_batch, self.n_gpus)
        y_chunks = torch.chunk(y_batch, self.n_gpus)

        # 2.  "GPU" Gradient Computation (gradient accumulation)
        self.optimizer.zero_grad()
        total_loss = 0
        for i in range(self.n_gpus):
            out = self.model(chunks[i])
            loss = F.mse_loss(out, y_chunks[i]) / self.n_gpus  # Mean
            loss.backward()  # Gradient 
            total_loss += loss.item() * self.n_gpus

        # 3. optimizer.step()  Gradient ( Mean)
        self.optimizer.step()
        return total_loss / self.n_gpus

#  Model 
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 20)
)

dp = DataParallelSimulator(model, n_gpus=4)

# Comparison:  Batch vs DDP 
x = torch.randn(64, 20)
y = torch.randn(64, 20)

loss = dp.step(x, y)
print(f"DDP  1 step loss: {loss:.4f}")
print(f"n_gpus=4, batch=64 →  GPU batch=16")
print(f"Effect Batch Magnitude: {dp.n_gpus * 16} = 64")


## 28.3 All-Reduce

 GPU GPU :
- **Ring All-Reduce**: $2(N-1)$ , $|\theta|/N$
- **Tree All-Reduce**:

: $O(|\theta|)$ per step. 7B モデル 14GB .

 → - .


In [ ]:
# All-Reduce 
def all_reduce_sum(grads_list):
    """ GPU Gradient Sum  GPU ."""
    # Sum
    total = sum(grads_list)
    #  GPU  Value 
    return [total.clone() for _ in grads_list]

# 4 GPU 
torch.manual_seed(0)
n_gpus = 4
local_grads = [torch.randn(5) for _ in range(n_gpus)]
print(" GPU  :")
for i, g in enumerate(local_grads):
    print(f"  GPU {i}: {g.numpy().round(3)}")

# All-Reduce (Mean)
reduced = all_reduce_sum(local_grads)
avg = [r / n_gpus for r in reduced]
print("\nAll-Reduce  ():")
for i, g in enumerate(avg):
    print(f"  GPU {i}: {g.numpy().round(3)}")
print("\n=>  GPU  Mean Gradient .")


## 28.4 PyTorch DDP 

 PyTorch :
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group('nccl', rank=rank, world_size=world_size)
model = model.to(rank)
model = DDP(model, device_ids=[rank])
# model(x)
```

Colab GPU GPU DDP , gradient accumulation .


In [ ]:
#  GPU DDP 
import time
from llm_math.bench import time_fn

#  モデル DDP vs   比較
def make_model(d=512):
    return nn.Sequential(
        nn.Linear(d, d * 2), nn.ReLU(),
        nn.Linear(d * 2, d), nn.ReLU(),
        nn.Linear(d, d)
    )

#   
torch.manual_seed(0)
model_single = make_model()
opt_single = torch.optim.AdamW(model_single.parameters(), lr=1e-3)

def step_single(x, y):
    opt_single.zero_grad()
    out = model_single(x)
    loss = F.mse_loss(out, y)
    loss.backward()
    opt_single.step()
    return loss

# DDP  (4 GPU,  1/4 )
torch.manual_seed(0)
model_ddp = make_model()
opt_ddp = torch.optim.AdamW(model_ddp.parameters(), lr=1e-3)

def step_ddp(x, y, n_gpus=4):
    opt_ddp.zero_grad()
    x_chunks = torch.chunk(x, n_gpus)
    y_chunks = torch.chunk(y, n_gpus)
    total_loss = 0
    for xc, yc in zip(x_chunks, y_chunks):
        out = model_ddp(xc)
        loss = F.mse_loss(out, yc) / n_gpus  # 
        loss.backward()  #  
        total_loss += loss.item() * n_gpus
    opt_ddp.step()
    return total_loss / n_gpus

# 時間 比較 (   )
d = 512
batch_size = 128
x = torch.randn(batch_size, d)
y = torch.randn(batch_size, d)

t_single = time_fn(step_single, x, y, device='cpu', warmup=2, repeat=5)
t_ddp = time_fn(step_ddp, x, y, device='cpu', warmup=2, repeat=5)
print(f"  Batch (B=128): {t_single['mean_ms']:.3f} ms")
print(f"DDP  (4 GPU,  B=32): {t_ddp['mean_ms']:.3f} ms")
print("\n=>  GPU DDP    (sequential ).")
print("     GPU    .")


## 28.5 Large Batch Training — LARS, LAMB

 学習 . 学習 :

**LARS** (Layer-wise Adaptive Rate Scaling):
$$\eta_\ell = \eta \cdot \frac{\|\theta_\ell\|}{\|\nabla \mathcal{L}_\ell\| + \lambda \|\theta_\ell\|}$$

**LAMB**: LARS + Adam. LLM 学習 .

## 28.6 -

バックプロパゲーション . 計算 ** All-Reduce **, バックプロパゲーション .

 時間 .

## 28.7 要点

| | |
|---|---|
| データ | モデル N, データ |
| All-Reduce | |
| | $N \cdot B_{\text{local}}$ |
| | $O(\|\theta\|)$ per step |
| LARS/LAMB | 学習 |

## 演習問題

1. 4-GPU DDP GPU データ 学習 .
2. 32, 64, 128, 256 学習 比較.
3. All-Reduce $O(\|\theta\|)$ .
4. - 速度 .
5. LARS 度 .

> 解答: `solutions/ch28_solutions.ipynb`
